# ManualMate — embed chunks on Colab and measure the baseline

Run the cells from top to bottom. Before you start:

1. **Runtime → Change runtime type → T4 GPU**, then Save.
2. Click the **key icon** (Secrets) on the left. Add a secret named `DATABASE_URL` with your Neon connection string (the same value as in your `.env`). Turn on **Notebook access**.
3. Have `data/chunks/chunks.jsonl` from your laptop ready to upload.

The notebook keeps nothing important: the vectors go to Neon, and the results file is downloaded to your laptop.

## 1. Check the GPU
Expected: a line with `Tesla T4`.

In [ ]:
!nvidia-smi -L

## 2. Get the latest code from GitHub

In [ ]:
import os
if not os.path.exists("/content/manualmate"):
    !git clone https://github.com/furkanhf/manualmate.git /content/manualmate
%cd /content/manualmate
!git pull
!git log --oneline -1

## 3. Install the libraries
Colab already has PyTorch with GPU support, so we do not install it.

In [ ]:
!pip install -q sentence-transformers==6.2.0 rank-bm25==0.2.2 "psycopg[binary]" pgvector python-dotenv
!pip install -q -e . --no-deps

## 4. Load the database secret
Expected: `DATABASE_URL loaded`. The value itself is never printed.

In [ ]:
from google.colab import userdata
os.environ["DATABASE_URL"] = userdata.get("DATABASE_URL")
print("DATABASE_URL loaded")

## 5. Upload chunks.jsonl
Click **Choose Files** and select `D:\\Projects\\manualmate\\data\\chunks\\chunks.jsonl`.
Expected: `2230 data/chunks/chunks.jsonl`.

In [ ]:
from google.colab import files
os.makedirs("data/chunks", exist_ok=True)
uploaded = files.upload()
name = next(iter(uploaded))
os.replace(name, "data/chunks/chunks.jsonl")
!wc -l data/chunks/chunks.jsonl

## 6. Embed all chunks and load them into Neon
Skip cells 5 and 6 if the chunks did not change (Neon keeps them).

The first run downloads bge-m3 (~2.3 GB, 1–2 min). Embedding takes about 1–2 min on a T4.
Expected last line: `Database now has 2230 chunks`.

In [ ]:
!python scripts/build_index.py

## 7. Measure retrieval (Recall@5 on 50 questions)
`routed` = parse the question → pick the manual → dense search in that manual.
- `--query-text issue`: search without brand and model words (default)
- `--query-text full`: search with the whole question
- `routed_hybrid`: the same, plus BM25 (RRF) inside the manual

In [ ]:
!python scripts/run_eval.py --method routed --query-text issue
!python scripts/run_eval.py --method routed --query-text full
!python scripts/run_eval.py --method routed_hybrid --query-text issue

## 8. Download the results files
Save them on your laptop in `D:\\Projects\\manualmate\\eval\\results\\`, then commit them.

In [ ]:
import glob
for path in sorted(glob.glob("eval/results/*.json")):
    print(path)
    files.download(path)